## Structured Output
Structured output = LLM output in a predefined, machine-readable format (like JSON)

In [50]:
import os
from langchain.chat_models import init_chat_model

llm = init_chat_model(model="groq:llama-3.1-8b-instant")
llm.profile

{'max_input_tokens': 131072,
 'max_output_tokens': 8192,
 'image_inputs': False,
 'audio_inputs': False,
 'video_inputs': False,
 'image_outputs': False,
 'audio_outputs': False,
 'video_outputs': False,
 'reasoning_output': False,
 'tool_calling': True}

### Pydantic

In [ ]:
from pydantic import BaseModel, Field

# Define the schema for the output
class Movies(BaseModel):
    title: str = Field(description="Title of the movie")
    year: int = Field(description="Year of release")
    rating: float = Field(description="Rating of the movie")
    genre: str = Field(description="Genre of the movie")
    

In [17]:
model_with_strucutre = llm.with_structured_output(Movies)

response = model_with_strucutre.invoke("Provide the details about the movie Avvenger: infinity war")
response

Movies(title='Avenger: infinity war', year=2018, rating=8.2, genre='Action, Adventure, Sci-Fi', youtube_trailer='https://www.youtube.com/watch?v=6ZfuNkozew4')

### Message output with Parsed Strucutre
#### getting both raw LLM message + parsed structured output

In [21]:

odel_with_parsed_struct = llm.with_structured_output(Movies, include_raw=True)


response = model_with_parsed_struct.invoke("Provide the details about the movie Avvenger: infinity war")
response

{'raw': AIMessage(content='', additional_kwargs={'tool_calls': [{'id': '6erdb3f8z', 'function': {'arguments': '{"genre":"Action, Adventure, Sci-Fi","rating":8,"title":"Avengers: Infinity War","year":2018,"youtube_trailer":"https://www.youtube.com/watch?v=uyyT7PQbWnU"}', 'name': 'Movies'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 63, 'prompt_tokens': 296, 'total_tokens': 359, 'completion_time': 0.118595361, 'completion_tokens_details': None, 'prompt_time': 0.019539611, 'prompt_tokens_details': None, 'queue_time': 0.077071228, 'total_time': 0.138134972}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_f757f4b0bf', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019d140e-e6fa-7de1-ab2f-ff850ad737b2-0', tool_calls=[{'name': 'Movies', 'args': {'genre': 'Action, Adventure, Sci-Fi', 'rating': 8, 'title': 'Avengers: Infinity War', 'year': 2018, 'youtube_trailer': 'https://

### Nested Strucutre

In [36]:
from typing import List
class Actor(BaseModel):
    name: str
    role: str
    
class TVShow(BaseModel):
    title: str
    year: int
    rating: float
    genre: str
    actors: List[Actor]

model_with_strucutre = llm.with_structured_output(TVShow)

response = model_with_strucutre.invoke("Provide the details about the TV Show Game Of Thrones")
response

TVShow(title='Game of Thrones', year=2011, rating=9.2, genre='Science Fiction, Drama, Fantasy', actors=[Actor(name='Peter Dinklage', role='Tywin Lannister'), Actor(name='Sophie Turner', role='Sansa Stark'), Actor(name='Maisie Williams', role='Arya Stark'), Actor(name='Kit Harington', role='Jon Snow'), Actor(name='Emilia Clarke', role='Daenerys Targaryen')])

### TypeDict

In [37]:
from typing_extensions import TypedDict, Annotated

In [46]:
class MoviesDict(TypedDict):
    """A movies class with details."""
    title: Annotated[str, "The title of the movie"]
    year: Annotated[int, "The year the movie was released"]
    rating: Annotated[float, "The rating of the movie out of 10"]
    genre: Annotated[str, "The genre of the movie"]


In [47]:
model_with_typedict = llm.with_structured_output(MoviesDict)

response = model_with_typedict.invoke("Tell me about the movies Interstellar")
response

{'genre': 'Science fiction',
 'rating': 8,
 'title': 'Interstellar',
 'year': 2014}

### Pydantic Structured output with Agents

In [54]:
from langchain.agents import create_agent

class ContactInfo(BaseModel):
    name: str
    email: str
    phone: str

agent = create_agent(
    model="groq:llama-3.1-8b-instant",
    response_format=ContactInfo
)

response = agent.invoke({"messages": "So I live at Washington DC, near street 32 and my name is Shiva. You can contact me at 9876543210 and if number is not reachable you can try shiv.aa@gmail.com"})
response["structured_response"]

ContactInfo(name='Shiva', email='shiv.aa@gmail.com', phone='9876543210')

### TypeDict output with Agents

In [55]:

class ContactInfo(TypedDict):
    name: str
    email: str
    phone: str

agent = create_agent(
    model="groq:llama-3.1-8b-instant",
    response_format=ContactInfo
)

response = agent.invoke({"messages": "So I live at Washington DC, near street 32 and my name is Shiva. You can contact me at 9876543210 and if number is not reachable you can try shiv.aa@gmail.com"})
response["structured_response"]

{'name': 'Shiva', 'email': 'shiv.aa@gmail.com', 'phone': '9876543210'}

## DataClass

In [ ]:
from dataclasses import dataclass

@dataclass
class ContactInfo:
    name: str
    email: str
    phone: str

agent = create_agent(
    model="groq:llama-3.1-8b-instant",
    response_format=ContactInfo
)

response = agent.invoke({"messages": "So I live at Washington DC, near street 32 and my name is Shiva. You can contact me at 9876543210 and if number is not reachable you can try shiv.aa@gmail.com"})
response["structured_response"]

ContactInfo2(name='Shiva', email='shiv.aa@gmail.com', phone='9876543210')